# Recovering hidden parameters from noisy HIV surveillance data

The [paper](https://doi.org/10.5556/j.tkjm.57.5817.2026) this package reproduces solves the
**forward** problem: put rates in, get trajectories out. This notebook solves the **inverse**
problem, which is the one an epidemiologist actually faces.

The reason is simple. Of the fourteen parameters in the model, the two that matter most for
policy are the two nobody can measure:

- $\beta$, the effective contact rate — a product of contact frequency and per-contact
  transmission probability, with no instrument that reads it off.
- $\alpha$, the rate at which diagnosed people start treatment — visible to programme
  registers only indirectly.

Both have to be inferred from their consequences. Below we generate data whose true answer we
know, hide the answer, recover it, and then ask the harder question: **how much of that
recovery should we believe?**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from hiv_drc import (
    DRC_2020,
    INITIAL_STATE,
    cost_surface,
    estimate_multistart,
    estimate_parameters,
    generate_observations,
    plotting,
    residuals,
    weight_vector,
)

print("truth we are going to hide from the estimator:")
print(f"  beta  = {DRC_2020.beta}")
print(f"  alpha = {DRC_2020.alpha}")

## 1. Generate observations

A surveillance system sees two of the six compartments: symptomatic cases $A$ and people on
treatment $T$. The other four — including *both* infected classes — are latent. Undiagnosed
infection is exactly what does not get counted, and that is what makes this an inverse problem
rather than a curve fit.

We take 31 annual observations over 30 years and add 5% proportional Gaussian noise.

In [ ]:
observations = generate_observations(
    DRC_2020,
    INITIAL_STATE,
    t_span=(0.0, 30.0),
    n_points=31,
    observed=("A", "T"),
    noise=0.05,
    noise_model="proportional",
    seed=20260830,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, name in zip(axes, observations.names):
    ax.plot(observations.t, observations.values[name], "o", color="0.3", label="observed")
    ax.plot(observations.t, observations.truth[name], "--", color="tab:blue", label="truth")
    ax.set_title(name)
    ax.set_xlabel("time (years)")
    ax.set_ylabel("millions")
    ax.legend(frameon=False)
plt.tight_layout()

## 2. The objective

The fit minimises a weighted sum of squared residuals over a bounded box:

$$
\hat{\theta} = \arg\min_{\theta \in [\ell,\,u]}
    \sum_{k}\sum_{i} w_k^2\left(y_k(t_i;\theta) - \hat{y}_{k,i}\right)^2
$$

Every evaluation costs a full integration of the six-compartment system.

Two details decide whether this works at all. **Weighting**: a sum of squares in raw units is
dominated by whichever compartment is largest — observe $S$ (≈ 88 million) beside $A$ (≈ 0.05
million) and the fit ignores $A$ completely. **Bounds**: unbounded, a trust-region step will
try $\beta < 0$, which turns the infection term into a source of susceptibles and the
integration diverges.

Note what the objective is *not* allowed to see. The weights come from
`observations.scale()`, computed from the noisy data alone — nothing about the truth leaks
into the estimate.

In [ ]:
w = weight_vector(observations, "scale")


def cost(beta, alpha):
    r = residuals((beta, alpha), ("beta", "alpha"), observations, w=w)
    return 0.5 * float(r @ r)


print(f"cost at the true parameters : {cost(DRC_2020.beta, DRC_2020.alpha):.6e}")
print(f"cost at a poor guess        : {cost(0.50, 0.25):.6e}")

## 3. Fit

The optimiser starts a quarter of the way up each parameter's box — $\beta = 0.5$,
$\alpha = 0.25$, deliberately far from the answer and derived from the *bounds*, not from the
baseline. Starting at the truth would make the whole exercise meaningless.

In [ ]:
fit = estimate_parameters(observations, fit=("beta", "alpha"))
print(fit.summary())

Note that the cost at the estimate is *below* the cost at the true parameters. That is not a
bug — it is what least squares does. The estimator fits this particular noise realisation, and
the sample optimum sits slightly away from the value that generated it. Any estimate whose
cost were *above* the truth would mean the optimiser stopped early.

In [ ]:
print(f"cost at estimate : {fit.cost:.8e}")
print(f"cost at truth    : {cost(DRC_2020.beta, DRC_2020.alpha):.8e}")

fig = plotting.plot_fit(observations, fit)

## 4. How much of this should we believe?

The residuals look like noise and the curves lie on top of each other, so it is tempting to
stop here. Don't. A small residual says the model *fits*; it says nothing about whether the
parameters are **identifiable**.

Look at the reported intervals: the 95% interval for $\alpha$ spans about ±27% of its
estimate, while the one for $\beta$ spans nearly ±90%. Mapping the objective shows why.

In [ ]:
grids = [
    np.linspace(0.8 * min(fit.ci95["beta"][0], DRC_2020.beta),
                1.15 * max(fit.ci95["beta"][1], DRC_2020.beta), 41),
    np.linspace(0.8 * min(fit.ci95["alpha"][0], DRC_2020.alpha),
                1.15 * max(fit.ci95["alpha"][1], DRC_2020.alpha), 41),
]
x, y, surface = cost_surface(observations, names=("beta", "alpha"), grids=grids)

fig = plotting.plot_cost_surface(x, y, surface, fit, ("beta", "alpha"))

The basin has a long flat floor along $\beta$. The reason is structural: $T$ responds to
$\alpha$ directly, while $\beta$ reaches the observed compartments only through the
*unobserved* $I_1 \to I_2$ chain. A wide range of contact rates produces observations that
differ by less than the measurement noise, and no optimiser can separate them. This is a
property of the experiment, not of the algorithm — the fix would be to observe more, not to
optimise harder.

## 5. Is this the global optimum?

A single local solve only ever finds the basin it was dropped into. Scattering starts across
the box is the cheapest available evidence.

In [ ]:
best = estimate_multistart(observations, fit=("beta", "alpha"), n_starts=8, seed=20260830)
for k, start in enumerate(best.starts):
    print(f"  start {k}: beta = {start['beta']:.6f}   alpha = {start['alpha']:.6f}")

spread = {
    name: max(s[name] for s in best.starts) - min(s[name] for s in best.starts)
    for name in best.names
}
print(f"\nspread across starts: {spread}")

## 6. The honest test: many noise realisations

One fit is one draw from a random variable. To say anything about the *estimator* rather than
this run, repeat it over independent noise realisations and check two things: is it biased,
and do the reported 95% intervals actually contain the truth 95% of the time?

This runs 40 fits per noise level and takes about ten seconds.

In [ ]:
rows = []
for eta in (0.02, 0.05, 0.10):
    errors = {"beta": [], "alpha": []}
    covered = []
    for seed in range(40):
        obs = generate_observations(noise=eta, seed=1000 + seed)
        f = estimate_parameters(obs)
        for name, value in f.relative_errors().items():
            errors[name].append(value)
        covered.append(all(f.covers_truth().values()))
    rows.append((eta, errors, np.mean(covered)))

print(f"{'noise':>6} {'beta bias':>11} {'beta sd':>9} {'alpha bias':>11} {'alpha sd':>9} {'coverage':>9}")
for eta, errors, coverage in rows:
    print(f"{100 * eta:5.0f}% {np.mean(errors['beta']):10.2f}% {np.std(errors['beta']):8.2f}% "
          f"{np.mean(errors['alpha']):10.2f}% {np.std(errors['alpha']):8.2f}% {100 * coverage:8.0f}%")

Two findings worth stating plainly:

1. **$\alpha$ is recovered well; $\beta$ is not.** At 5% noise, $\alpha$ scatters by about 11%
   and $\beta$ by about 22%. That matches the shape of the objective landscape above.
2. **The intervals are calibrated only where the linearisation holds.** At 5% noise coverage
   is 95%, exactly nominal. At 10% it falls to around 88% — the asymptotic Wald intervals
   become optimistic once the model's nonlinearity matters across the width of the interval.
   If you need intervals at that noise level, bootstrap or profile the likelihood instead.

## What is *not* estimated

Twelve of the fourteen parameters were held fixed at published values. Those are assumptions,
not results, and being wrong about them biases what we did estimate:

In [ ]:
clean = generate_observations(noise=0.0)
honest = estimate_parameters(clean)
biased = estimate_parameters(clean, baseline=DRC_2020.replace(sigma2=0.12))

print("fitting noise-free data with sigma2 held at the right value:")
print(f"  alpha = {honest.estimates['alpha']:.6f}  (truth {DRC_2020.alpha})")
print("fitting the same data with sigma2 held at a wrong value (0.12 instead of 0.06):")
print(f"  alpha = {biased.estimates['alpha']:.6f}  "
      f"({biased.relative_errors()['alpha']:+.1f}% off)")

The data are noise-free and the optimiser converges perfectly — and the answer is still wrong,
because the question was wrong. This is why the fitted set is kept small and the rest has to
be defensible. It is also the failure mode that a synthetic-data workflow catches and a
real-data workflow silently hides.